In [ ]:
# CELL 5
import os
import shutil
import time
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

CPU_COUNT = os.cpu_count() or 8
BASE_DIR = Path(os.environ.get("DATA_LABEL_BASE", "/home/vietnam/voice_to_text/data/data_label_auto"))
SEG_DIR = BASE_DIR / "segments_vad"
DENOISE_DIR = BASE_DIR / "segments_denoised"
MANIFEST_DIR = BASE_DIR / "manifests"

DENOISE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

vad_manifest = MANIFEST_DIR / "vad_segments.csv"
denoise_manifest = MANIFEST_DIR / "denoised_segments.csv"

# Local staging to reduce storage I/O during compute
default_stage_root = Path("/dev/shm/denoise_stage") if Path("/dev/shm").exists() else (BASE_DIR / ".denoise_stage")
LOCAL_STAGE_ROOT = Path(os.environ.get("DENOISE_STAGE_ROOT", str(default_stage_root)))
LOCAL_IN_DIR = LOCAL_STAGE_ROOT / "in"
LOCAL_OUT_DIR = LOCAL_STAGE_ROOT / "out"
LOCAL_IN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not vad_manifest.exists():
    raise FileNotFoundError(f"Missing VAD manifest: {vad_manifest}")

df_vad = pd.read_csv(vad_manifest)
if df_vad.empty:
    raise RuntimeError("vad_segments.csv is empty. Run CELL 5 first.")

SR = 16000
FORCE_REPROCESS = False

# Auto-run controls
AUTO_RUN_UNTIL_DONE = True
MAX_AUTO_ROUNDS = int(os.environ.get("DENOISE_MAX_AUTO_ROUNDS", 999))
SLEEP_BETWEEN_ROUNDS_SEC = float(os.environ.get("DENOISE_SLEEP_BETWEEN_ROUNDS_SEC", 1))

# Per-round safety window
MAX_FILES_PER_ROUND = int(os.environ.get("DENOISE_MAX_FILES_PER_ROUND", 6000))
MAX_RUN_MINUTES_PER_ROUND = float(os.environ.get("DENOISE_MAX_RUN_MINUTES_PER_ROUND", 30))
PREFETCH_FILES_PER_ROUND = int(os.environ.get("DENOISE_PREFETCH_FILES_PER_ROUND", 800))
SAVE_EVERY = int(os.environ.get("DENOISE_SAVE_EVERY", 200))

# Resume mode
RESUME_BY_MANIFEST_ONLY = True
VERIFY_OUTPUT_EXISTS_FOR_RESUME = False  # set True only for recovery scan (slow)

# CPU controls
DENOISE_TORCH_THREADS = int(os.environ.get("DENOISE_TORCH_THREADS", max(1, min(48, CPU_COUNT - 4))))
DENOISE_INTEROP_THREADS = int(os.environ.get("DENOISE_INTEROP_THREADS", max(1, min(8, DENOISE_TORCH_THREADS // 4 or 1))))
BATCH_SIZE = int(os.environ.get("DENOISE_BATCH_SIZE", max(12, min(64, DENOISE_TORCH_THREADS))))

# Retry I/O
IO_RETRIES = 3
IO_SLEEP_SEC = 1.5

# Torch spectral denoise config
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
NOISE_PROFILE_SEC = 0.5
SPECTRAL_SUB_FACTOR = 1.0
MASK_FLOOR = 0.06

DENOISE_DEVICE = torch.device("cpu")  # forced CPU mode

torch.set_num_threads(max(1, DENOISE_TORCH_THREADS))
try:
    torch.set_num_interop_threads(max(1, DENOISE_INTEROP_THREADS))
except Exception:
    pass

print("Denoise device:", DENOISE_DEVICE, "(forced)")
print("BASE_DIR:", BASE_DIR)
print("LOCAL_STAGE_ROOT:", LOCAL_STAGE_ROOT)
print("CPU_COUNT:", CPU_COUNT)
print("DENOISE_TORCH_THREADS:", DENOISE_TORCH_THREADS, "| DENOISE_INTEROP_THREADS:", DENOISE_INTEROP_THREADS)
print("BATCH_SIZE:", BATCH_SIZE, "| PREFETCH_FILES_PER_ROUND:", PREFETCH_FILES_PER_ROUND)
print("FORCE_REPROCESS:", FORCE_REPROCESS)
print("AUTO_RUN_UNTIL_DONE:", AUTO_RUN_UNTIL_DONE)
print("MAX_AUTO_ROUNDS:", MAX_AUTO_ROUNDS)
print("MAX_FILES_PER_ROUND:", MAX_FILES_PER_ROUND)
print("MAX_RUN_MINUTES_PER_ROUND:", MAX_RUN_MINUTES_PER_ROUND)
print("SAVE_EVERY:", SAVE_EVERY)

torch.set_grad_enabled(False)
window = torch.hann_window(WIN_LENGTH, device=DENOISE_DEVICE)


def is_drive_timeout_error(msg: str) -> bool:
    m = (msg or "").lower()
    keys = [
        "input/output error",
        "transport endpoint is not connected",
        "stale file handle",
        "timed out",
        "connection reset",
        "operation not permitted",
    ]
    return any(k in m for k in keys)


def safe_copy(src: Path, dst: Path):
    last_e = None
    dst.parent.mkdir(parents=True, exist_ok=True)
    for _ in range(IO_RETRIES):
        try:
            shutil.copy2(src, dst)
            return
        except Exception as e:
            last_e = e
            time.sleep(IO_SLEEP_SEC)
    raise last_e


def safe_read_audio(path: Path):
    last_e = None
    for _ in range(IO_RETRIES):
        try:
            return sf.read(path, dtype="float32")
        except Exception as e:
            last_e = e
            time.sleep(IO_SLEEP_SEC)
    raise last_e


def safe_write_audio(path: Path, audio_np: np.ndarray, sr: int):
    last_e = None
    path.parent.mkdir(parents=True, exist_ok=True)
    for _ in range(IO_RETRIES):
        try:
            sf.write(path, audio_np, sr)
            return
        except Exception as e:
            last_e = e
            time.sleep(IO_SLEEP_SEC)
    raise last_e


def read_and_prepare_audio_local(path: Path):
    audio_np, sr_in = safe_read_audio(path)
    x = np.asarray(audio_np, dtype=np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if sr_in != SR:
        x = librosa.resample(x, orig_sr=sr_in, target_sr=SR).astype(np.float32)
    if len(x) == 0:
        return np.zeros(0, dtype=np.float32)
    return np.clip(x, -1.0, 1.0)


def denoise_batch_torch(batch_audio_np):
    if not batch_audio_np:
        return []

    lengths = [len(x) for x in batch_audio_np]
    max_len = max(lengths)
    if max_len == 0:
        return [np.zeros(0, dtype=np.float32) for _ in batch_audio_np]

    batch_np = np.zeros((len(batch_audio_np), max_len), dtype=np.float32)
    for i, x in enumerate(batch_audio_np):
        if len(x) > 0:
            batch_np[i, : len(x)] = x

    wave = torch.from_numpy(batch_np).to(DENOISE_DEVICE)

    with torch.no_grad():
        spec = torch.stft(
            wave,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH,
            window=window,
            return_complex=True,
        )

        mag = torch.abs(spec)
        total_frames = mag.shape[-1]
        noise_frames = max(4, int(NOISE_PROFILE_SEC * SR / HOP_LENGTH))
        noise_frames = min(noise_frames, total_frames)

        noise_profile = mag[:, :, :noise_frames].mean(dim=2, keepdim=True)
        clean_mag = torch.clamp(mag - SPECTRAL_SUB_FACTOR * noise_profile, min=0.0)
        mask = clean_mag / (mag + 1e-8)
        mask = torch.clamp(mask, min=MASK_FLOOR, max=1.0)

        enhanced_spec = spec * mask
        enhanced = torch.istft(
            enhanced_spec,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH,
            window=window,
            length=max_len,
        )
        enhanced = torch.clamp(enhanced, -1.0, 1.0)

    enhanced_np = enhanced.detach().cpu().numpy().astype(np.float32)
    return [enhanced_np[i, : lengths[i]] for i in range(len(lengths))]


def load_existing_map():
    row_map = {}
    if not denoise_manifest.exists():
        return row_map

    try:
        df = pd.read_csv(denoise_manifest)
        if df.empty or "segment_path" not in df.columns:
            return row_map

        df = df.drop_duplicates(subset=["segment_path"], keep="last")

        if VERIFY_OUTPUT_EXISTS_FOR_RESUME:
            keep_rows = []
            for r in df.to_dict("records"):
                dp = str(r.get("denoised_path", "")).strip()
                if dp and Path(dp).exists():
                    keep_rows.append(r)
            df = pd.DataFrame(keep_rows) if keep_rows else pd.DataFrame(columns=df.columns)

        for r in df.to_dict("records"):
            row_map[str(r["segment_path"])] = r
    except Exception as e:
        print(f"Warning: cannot read old denoised manifest: {e}")

    return row_map


def save_manifest_checkpoint(row_map):
    if not row_map:
        return
    mdf = pd.DataFrame(list(row_map.values()))
    if not mdf.empty and "segment_path" in mdf.columns:
        mdf = mdf.drop_duplicates(subset=["segment_path"], keep="last")
    tmp = denoise_manifest.with_suffix(".tmp.csv")
    mdf.to_csv(tmp, index=False)
    tmp.replace(denoise_manifest)


def clear_local_stage():
    for p in LOCAL_IN_DIR.glob("*.wav"):
        try:
            p.unlink()
        except Exception:
            pass
    for p in LOCAL_OUT_DIR.glob("*.wav"):
        try:
            p.unlink()
        except Exception:
            pass


def build_pending(records, processed_set):
    pending = []
    for row in records:
        segp = str(row["segment_path"])
        if not FORCE_REPROCESS and RESUME_BY_MANIFEST_ONLY and segp in processed_set:
            continue

        in_drive = Path(segp)
        out_drive = DENOISE_DIR / in_drive.name
        if FORCE_REPROCESS:
            pending.append((row, in_drive, out_drive))
        else:
            if segp in processed_set:
                continue
            pending.append((row, in_drive, out_drive))
    return pending


def run_one_round(round_id: int, pending_subset, existing_map):
    processed_now = 0
    failed_now = 0
    drive_broken = False
    round_start = time.time()

    if not pending_subset:
        return processed_now, failed_now, drive_broken

    pbar = tqdm(total=len(pending_subset), desc=f"Denoise[R{round_id}]")
    cursor = 0

    while cursor < len(pending_subset):
        if MAX_RUN_MINUTES_PER_ROUND and ((time.time() - round_start) >= MAX_RUN_MINUTES_PER_ROUND * 60):
            print(f"Round {round_id}: reached MAX_RUN_MINUTES_PER_ROUND -> checkpoint and continue next round.")
            break

        round_tasks = pending_subset[cursor : cursor + max(1, int(PREFETCH_FILES_PER_ROUND))]
        staged = []

        # 1) Prefetch from Drive -> local
        for row, in_drive, out_drive in round_tasks:
            local_in = LOCAL_IN_DIR / in_drive.name
            local_out = LOCAL_OUT_DIR / in_drive.name
            try:
                safe_copy(in_drive, local_in)
                staged.append((row, in_drive, out_drive, local_in, local_out))
            except Exception as e:
                msg = str(e)
                print(f"Round {round_id} prefetch failed: {in_drive.name} -> {msg}")
                if is_drive_timeout_error(msg):
                    drive_broken = True
                    break
                failed_now += 1
                cursor += 1
                pbar.update(1)

        if drive_broken:
            break

        # 2) Denoise local staged files
        i = 0
        while i < len(staged):
            if MAX_RUN_MINUTES_PER_ROUND and ((time.time() - round_start) >= MAX_RUN_MINUTES_PER_ROUND * 60):
                print(f"Round {round_id}: reached MAX_RUN_MINUTES_PER_ROUND during processing.")
                break

            chunk = staged[i : i + max(1, int(BATCH_SIZE))]
            try:
                batch_audio = [read_and_prepare_audio_local(local_in) for _, _, _, local_in, _ in chunk]
                batch_out = denoise_batch_torch(batch_audio)

                for (row, _, out_drive, local_in, local_out), y in zip(chunk, batch_out):
                    try:
                        safe_write_audio(local_out, y, SR)
                        safe_copy(local_out, out_drive)

                        row_cp = dict(row)
                        row_cp["denoised_path"] = str(out_drive)
                        row_cp["duration_sec"] = round(len(y) / SR, 3)
                        existing_map[str(row_cp["segment_path"])] = row_cp
                        processed_now += 1
                    except Exception as ew:
                        failed_now += 1
                        msgw = str(ew)
                        print(f"Round {round_id} write/sync failed {out_drive.name}: {msgw}")
                        if is_drive_timeout_error(msgw):
                            drive_broken = True
                            break
                    finally:
                        try:
                            local_in.unlink(missing_ok=True)
                        except Exception:
                            pass
                        try:
                            local_out.unlink(missing_ok=True)
                        except Exception:
                            pass
                        pbar.update(1)
                        cursor += 1

                if drive_broken:
                    break

                if SAVE_EVERY > 0 and processed_now > 0 and processed_now % SAVE_EVERY == 0:
                    save_manifest_checkpoint(existing_map)
                    tqdm.write(f"Round {round_id} checkpoint: processed_now={processed_now}, manifest_rows={len(existing_map)}")

                i += len(chunk)

            except Exception as e:
                failed_now += 1
                msg = str(e)
                print(f"Round {round_id} batch error at idx={cursor}: {msg}")
                if is_drive_timeout_error(msg):
                    drive_broken = True
                    break

                for _, _, _, local_in, local_out in chunk:
                    try:
                        local_in.unlink(missing_ok=True)
                    except Exception:
                        pass
                    try:
                        local_out.unlink(missing_ok=True)
                    except Exception:
                        pass
                    pbar.update(1)
                    cursor += 1
                i += len(chunk)

        if drive_broken:
            break

    pbar.close()
    save_manifest_checkpoint(existing_map)
    clear_local_stage()

    return processed_now, failed_now, drive_broken


records = df_vad.to_dict("records")
overall_processed = 0
overall_failed = 0
stop_due_to_drive = False

for round_id in range(1, int(MAX_AUTO_ROUNDS) + 1):
    existing_map = load_existing_map()
    processed_set = set(existing_map.keys()) if not FORCE_REPROCESS else set()

    pending_all = build_pending(records, processed_set)
    total_vad = len(records)
    done_now = len(processed_set)
    pct = 100.0 * done_now / max(total_vad, 1)

    print("")
    print(f"=== Round {round_id} Plan ===")
    print("Total VAD rows      :", total_vad)
    print("Already in manifest :", done_now, f"({pct:.2f}%)")
    print("Pending total       :", len(pending_all))

    if len(pending_all) == 0:
        print("All done. No pending files.")
        break

    pending_subset = pending_all[: int(MAX_FILES_PER_ROUND)] if MAX_FILES_PER_ROUND is not None else pending_all
    print("Pending this round  :", len(pending_subset))

    proc, fail, drive_broken = run_one_round(round_id, pending_subset, existing_map)
    overall_processed += proc
    overall_failed += fail

    # reload for exact latest stats
    latest_map = load_existing_map()
    latest_done = len(latest_map)
    latest_pct = 100.0 * latest_done / max(total_vad, 1)

    print("")
    print(f"=== Round {round_id} Summary ===")
    print("Processed this round:", proc)
    print("Failed this round   :", fail)
    print("Manifest rows total :", latest_done, f"({latest_pct:.2f}%)")

    if drive_broken:
        stop_due_to_drive = True
        print("Status: stopped due to Drive timeout/I/O. Remount Drive then rerun CELL 6.")
        break

    if not AUTO_RUN_UNTIL_DONE:
        print("AUTO_RUN_UNTIL_DONE=False -> stop after one round.")
        break

    if proc == 0:
        print("No progress in this round -> stop to avoid infinite loop.")
        break

    if SLEEP_BETWEEN_ROUNDS_SEC > 0:
        time.sleep(SLEEP_BETWEEN_ROUNDS_SEC)

# final summary
final_map = load_existing_map()
final_df = pd.DataFrame(list(final_map.values())) if final_map else pd.DataFrame()

print("")
print("=== Denoise Final Summary ===")
print("Processed in this cell run:", overall_processed)
print("Failed in this cell run   :", overall_failed)
print("Manifest rows total       :", len(final_df))
print("Saved manifest            :", denoise_manifest)
if stop_due_to_drive:
    print("Final status              : paused by Drive timeout (needs remount + rerun).")
else:
    print("Final status              : finished current auto-run cycle.")

if not final_df.empty:
    display(final_df.head())




In [ ]:
# CELL 6
# ===== TEN-VAD (CPU) + FULL FOLDER + RESUME + PARALLEL FILE WORKERS =====
import os
import subprocess
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm


def ensure_ten_vad():
    try:
        from ten_vad import TenVad as _TenVad
        return _TenVad
    except ModuleNotFoundError:
        print("ten_vad not found. Installing from GitHub...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-U",
                "--force-reinstall",
                "git+https://github.com/TEN-framework/ten-vad.git",
            ],
            check=True,
        )
        from ten_vad import TenVad as _TenVad
        return _TenVad


TenVad = ensure_ten_vad()


def patch_ten_vad_destructor(cls):
    original_del = getattr(cls, "__del__", None)
    if original_del is None:
        return

    def _safe_del(self):
        if not hasattr(self, "vad_library") or not hasattr(self, "vad_handler"):
            return
        try:
            original_del(self)
        except Exception:
            pass

    cls.__del__ = _safe_del


patch_ten_vad_destructor(TenVad)

# Paths
CPU_COUNT = os.cpu_count() or 8
BASE_DIR = Path(os.environ.get("DATA_LABEL_BASE", "/home/vietnam/voice_to_text/data/data_label_auto"))
RAW_DIR = Path(os.environ.get("RAW_WAV_DIR", str(BASE_DIR / "raw")))
SEG_DIR = Path(os.environ.get("SEGMENTS_VAD_DIR", str(BASE_DIR / "segments_vad")))
MANIFEST_DIR = Path(os.environ.get("MANIFEST_DIR", str(BASE_DIR / "manifests")))
VAD_MANIFEST = Path(os.environ.get("VAD_MANIFEST_PATH", str(MANIFEST_DIR / "vad_segments.csv")))

SEG_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# Core config
SR = 16000
MIN_SEG_SEC = 8.0
MAX_SEG_SEC = 20.0
PAD_SEC = 0.15

# If VAD cuts too fragmented, join nearby short chunks until min duration.
PACK_SHORT_MAX_GAP_SEC = 1.2
FORCE_SHORT_TO_NEXT_GAP_SEC = 3.0

# Ten-VAD config
VAD_HOP_SIZE = 256
VAD_THRESHOLD = 0.5
MIN_SILENCE_MS = 250
MERGE_NEARBY_GAP_SEC = 0.6

# Parallelism (CPU only)
VAD_PARALLEL = True
VAD_NUM_WORKERS = int(os.environ.get("VAD_NUM_WORKERS", max(1, min(48, CPU_COUNT // 2))))
VAD_EXECUTOR = os.environ.get("VAD_EXECUTOR", "thread").strip().lower()  # thread | process
if VAD_EXECUTOR not in {"process", "thread"}:
    VAD_EXECUTOR = "thread"

# Long-tail mitigation: process long files first + limit in-flight futures.
VAD_SORT_LONG_FIRST = os.environ.get("VAD_SORT_LONG_FIRST", "1").strip().lower() in {"1", "true", "yes", "y"}
VAD_MAX_IN_FLIGHT = int(os.environ.get("VAD_MAX_IN_FLIGHT", max(VAD_NUM_WORKERS * 4, VAD_NUM_WORKERS)))

# Resume / checkpoint config
RESUME_VAD = True
CHECKPOINT_EVERY_FILES = int(os.environ.get("VAD_CHECKPOINT_EVERY_FILES", 20))
DROP_DUP_SEGMENT_PATH = True

print("VAD engine: Ten-VAD (CPU only)")
print("CUDA available:", torch.cuda.is_available(), "-> Ten-VAD does not use CUDA")
if torch.cuda.is_available():
    print("GPU reserved for other steps:", torch.cuda.get_device_name(0))
print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("SEG_DIR:", SEG_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("CPU_COUNT:", CPU_COUNT)
print("VAD_PARALLEL:", VAD_PARALLEL, "| VAD_NUM_WORKERS:", VAD_NUM_WORKERS, "| VAD_EXECUTOR:", VAD_EXECUTOR)
print("VAD_SORT_LONG_FIRST:", VAD_SORT_LONG_FIRST, "| VAD_MAX_IN_FLIGHT:", VAD_MAX_IN_FLIGHT)
print("RESUME_VAD:", RESUME_VAD, "| CHECKPOINT_EVERY_FILES:", CHECKPOINT_EVERY_FILES)
print("VAD_MANIFEST:", VAD_MANIFEST)


def collect_input_wavs(raw_dir: Path):
    """
    Collect all .wav files and de-duplicate by stem key.
    If both xxx.wav and xxx_16k.wav exist, prefer xxx_16k.wav.
    """
    all_wavs = sorted(raw_dir.rglob("*.wav"))
    by_key = {}

    for p in all_wavs:
        key = p.stem[:-4] if p.stem.endswith("_16k") else p.stem
        cur = by_key.get(key)
        if cur is None:
            by_key[key] = p
            continue
        if p.stem.endswith("_16k") and not cur.stem.endswith("_16k"):
            by_key[key] = p

    return sorted(by_key.values())


def split_long_region(audio, start_sec, end_sec, min_seg=MIN_SEG_SEC, max_seg=MAX_SEG_SEC, sr=SR):
    out = []
    s = start_sec
    while (end_sec - s) > max_seg:
        target = s + max_seg
        left = int(max(s + min_seg, target - 2.0) * sr)
        right = int(min(end_sec, target + 2.0) * sr)
        if right <= left:
            cut = target
        else:
            win = audio[left:right]
            frame = int(0.03 * sr)
            hop = int(0.01 * sr)
            if len(win) < frame:
                cut = target
            else:
                energies = [
                    float(np.mean(win[i : i + frame] ** 2))
                    for i in range(0, len(win) - frame + 1, hop)
                ]
                cut = (left + np.argmin(energies) * hop + int(0.015 * sr)) / sr
        if (cut - s) < min_seg:
            cut = s + min_seg
        if (cut - s) > max_seg:
            cut = s + max_seg
        out.append((s, cut))
        s = cut
    if end_sec - s >= min_seg:
        out.append((s, end_sec))
    return out


def save_segments(audio, segments, src_name, out_dir=SEG_DIR, sr=SR):
    rows = []
    for idx, (s, e) in enumerate(segments):
        clip = audio[int(s * sr) : int(e * sr)]
        if len(clip) == 0:
            continue
        out_path = out_dir / f"{src_name}__seg_{idx:05d}_{s:.2f}_{e:.2f}.wav"
        sf.write(out_path, clip, sr)
        rows.append(
            {
                "src": src_name,
                "segment_path": str(out_path),
                "duration_sec": round(e - s, 3),
            }
        )
    return rows


def to_pcm16(audio_f32: np.ndarray) -> np.ndarray:
    x = np.asarray(audio_f32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if x.dtype == np.int16:
        return x
    if np.issubdtype(x.dtype, np.floating):
        x = np.clip(x, -1.0, 1.0)
        return (x * 32767.0).astype(np.int16)
    return x.astype(np.int16)


def get_speech_regions_tenvad(
    audio_f32: np.ndarray,
    sr: int,
    hop_size: int = VAD_HOP_SIZE,
    threshold: float = VAD_THRESHOLD,
    min_silence_duration_ms: int = MIN_SILENCE_MS,
):
    if sr != SR:
        audio_f32 = librosa.resample(np.asarray(audio_f32, dtype=np.float32), orig_sr=sr, target_sr=SR)
        sr = SR

    pcm = to_pcm16(audio_f32)
    num_frames = len(pcm) // hop_size
    if num_frames <= 0:
        return [], audio_f32

    vad = TenVad(hop_size=hop_size, threshold=threshold)
    flags = np.zeros(num_frames, dtype=np.int8)

    for i in range(num_frames):
        frame = pcm[i * hop_size : (i + 1) * hop_size]
        _, out_flag = vad.process(frame)
        flags[i] = 1 if out_flag else 0

    speech_idx = np.flatnonzero(flags == 1)
    if len(speech_idx) == 0:
        return [], audio_f32

    frame_sec = hop_size / sr
    max_gap_frames = max(1, int(round((min_silence_duration_ms / 1000.0) / frame_sec)))

    regions = []
    start = int(speech_idx[0])
    prev = int(speech_idx[0])

    for idx in speech_idx[1:]:
        idx = int(idx)
        if idx - prev <= max_gap_frames:
            prev = idx
            continue

        regions.append((start * frame_sec, min(len(audio_f32), (prev + 1) * hop_size) / sr))
        start = idx
        prev = idx

    regions.append((start * frame_sec, min(len(audio_f32), (prev + 1) * hop_size) / sr))
    return regions, audio_f32


def merge_nearby_regions(regions, max_gap_sec: float = MERGE_NEARBY_GAP_SEC):
    if not regions:
        return []

    regions = sorted(regions, key=lambda x: x[0])
    merged = [[float(regions[0][0]), float(regions[0][1])]]

    for s, e in regions[1:]:
        s = float(s)
        e = float(e)
        if s - merged[-1][1] <= max_gap_sec:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    return [(s, e) for s, e in merged]


def pack_short_regions_to_min_duration(
    regions,
    min_seg=MIN_SEG_SEC,
    max_join_gap_sec=PACK_SHORT_MAX_GAP_SEC,
    force_short_to_next_gap_sec=FORCE_SHORT_TO_NEXT_GAP_SEC,
):
    if not regions:
        return []

    regions = sorted(regions, key=lambda x: x[0])
    packed = []
    i = 0

    while i < len(regions):
        start = float(regions[i][0])
        end = float(regions[i][1])
        j = i

        while (end - start) < min_seg and (j + 1) < len(regions):
            next_s, next_e = float(regions[j + 1][0]), float(regions[j + 1][1])
            gap = next_s - end

            allow_normal = gap <= max_join_gap_sec
            allow_rescue = gap <= force_short_to_next_gap_sec
            if not (allow_normal or allow_rescue):
                break

            end = next_e
            j += 1

        packed.append((start, end))
        i = j + 1

    return packed


def process_one_wav(wav_path: Path):
    out = {
        "rows": [],
        "raw_regions": 0,
        "raw_sec": 0.0,
        "merged_regions": 0,
        "merged_sec": 0.0,
        "padded_regions": 0,
        "padded_sec": 0.0,
        "packed_regions": 0,
        "packed_sec": 0.0,
        "packed_durations": [],
        "dropped_short_regions": 0,
        "dropped_short_sec": 0.0,
        "kept_lenfilter_sec": 0.0,
        "error": None,
        "file": wav_path.name,
        "src": wav_path.stem,
        "final_segments": 0,
    }

    try:
        audio_f32, sr_sf = sf.read(wav_path, dtype="float32")
        if isinstance(audio_f32, np.ndarray) and audio_f32.ndim == 2:
            audio_f32 = audio_f32.mean(axis=1)

        speech_regions, audio_16k = get_speech_regions_tenvad(
            audio_f32=audio_f32,
            sr=sr_sf,
            hop_size=VAD_HOP_SIZE,
            threshold=VAD_THRESHOLD,
            min_silence_duration_ms=MIN_SILENCE_MS,
        )
        out["raw_regions"] = len(speech_regions)
        out["raw_sec"] = sum(max(0.0, e - s) for s, e in speech_regions)

        merged_regions = merge_nearby_regions(speech_regions, max_gap_sec=MERGE_NEARBY_GAP_SEC)
        out["merged_regions"] = len(merged_regions)
        out["merged_sec"] = sum(max(0.0, e - s) for s, e in merged_regions)

        padded_regions = []
        for s, e in merged_regions:
            s = max(0.0, s - PAD_SEC)
            e = min(len(audio_16k) / SR, e + PAD_SEC)
            dur = e - s
            if dur <= 0:
                continue
            padded_regions.append((s, e))
            out["padded_regions"] += 1
            out["padded_sec"] += dur

        packed_regions = pack_short_regions_to_min_duration(
            padded_regions,
            min_seg=MIN_SEG_SEC,
            max_join_gap_sec=PACK_SHORT_MAX_GAP_SEC,
            force_short_to_next_gap_sec=FORCE_SHORT_TO_NEXT_GAP_SEC,
        )
        out["packed_regions"] = len(packed_regions)
        out["packed_sec"] = sum(max(0.0, e - s) for s, e in packed_regions)
        out["packed_durations"] = [max(0.0, e - s) for s, e in packed_regions]

        final_segments = []
        for s, e in packed_regions:
            dur = e - s
            if dur < MIN_SEG_SEC:
                out["dropped_short_regions"] += 1
                out["dropped_short_sec"] += dur
                continue

            if dur <= MAX_SEG_SEC:
                final_segments.append((s, e))
            else:
                final_segments.extend(split_long_region(audio_16k, s, e))

        out["kept_lenfilter_sec"] = sum(max(0.0, e - s) for s, e in final_segments)
        out["rows"] = save_segments(audio_16k, final_segments, wav_path.stem)
        out["final_segments"] = len(final_segments)

    except Exception as e:
        out["error"] = str(e)

    return out


def dedupe_manifest_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    if DROP_DUP_SEGMENT_PATH and "segment_path" in out.columns:
        out = out.drop_duplicates(subset=["segment_path"], keep="first")
    return out


def save_manifest(rows):
    if not rows:
        return pd.DataFrame(columns=["src", "segment_path", "duration_sec"])
    df = pd.DataFrame(rows)
    df = dedupe_manifest_df(df)
    tmp = VAD_MANIFEST.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(VAD_MANIFEST)
    return df


# Load resume state
existing_rows = []
processed_src = set()
if RESUME_VAD and VAD_MANIFEST.exists():
    try:
        old_df = pd.read_csv(VAD_MANIFEST)
        old_df = dedupe_manifest_df(old_df)
        if "src" in old_df.columns:
            processed_src = set(old_df["src"].astype(str).tolist())
            existing_rows = old_df.to_dict("records")
            print(f"Resume manifest found: {len(old_df)} segments from {len(processed_src)} src files")
        else:
            print("Resume manifest has no 'src' column; resume disabled for this run.")
    except Exception as e:
        print(f"Failed reading old manifest, run from scratch in-memory: {e}")

wavs_all = collect_input_wavs(RAW_DIR)
pending_wavs = [w for w in wavs_all if w.stem not in processed_src]

if pending_wavs and VAD_SORT_LONG_FIRST:
    dur_cache = {}

    def _wav_dur_sec(p):
        k = str(p)
        if k in dur_cache:
            return dur_cache[k]
        try:
            d = float(sf.info(str(p)).duration)
        except Exception:
            d = -1.0
        dur_cache[k] = d
        return d

    pending_wavs = sorted(pending_wavs, key=_wav_dur_sec, reverse=True)

print(f"Total input wavs: {len(wavs_all)}")
print(f"Already processed src (resume): {len(processed_src)}")
print(f"Pending wavs this run: {len(pending_wavs)}")
if pending_wavs and VAD_SORT_LONG_FIRST:
    try:
        print("Top pending duration (sec):", round(float(sf.info(str(pending_wavs[0])).duration), 2))
    except Exception:
        pass

all_rows = list(existing_rows)
added_rows_sec = 0.0
added_rows_count = 0
added_src_count = 0
files_since_ckpt = 0

# Debug counters for this run only
raw_speech_regions_total = 0
raw_speech_sec_total = 0.0
merged_speech_regions_total = 0
merged_speech_sec_total = 0.0
padded_regions_total = 0
padded_speech_sec_total = 0.0
packed_regions_total = 0
packed_speech_sec_total = 0.0
packed_region_durations = []
dropped_short_regions_total = 0
dropped_short_sec_total = 0.0
kept_after_lenfilter_sec_total = 0.0


def handle_result(res, idx_done, total):
    global raw_speech_regions_total, raw_speech_sec_total
    global merged_speech_regions_total, merged_speech_sec_total
    global padded_regions_total, padded_speech_sec_total
    global packed_regions_total, packed_speech_sec_total
    global dropped_short_regions_total, dropped_short_sec_total
    global kept_after_lenfilter_sec_total
    global added_rows_sec, added_rows_count, added_src_count, files_since_ckpt

    raw_speech_regions_total += res["raw_regions"]
    raw_speech_sec_total += res["raw_sec"]
    merged_speech_regions_total += res["merged_regions"]
    merged_speech_sec_total += res["merged_sec"]
    padded_regions_total += res["padded_regions"]
    padded_speech_sec_total += res["padded_sec"]
    packed_regions_total += res["packed_regions"]
    packed_speech_sec_total += res["packed_sec"]
    packed_region_durations.extend(res["packed_durations"])
    dropped_short_regions_total += res["dropped_short_regions"]
    dropped_short_sec_total += res["dropped_short_sec"]
    kept_after_lenfilter_sec_total += res["kept_lenfilter_sec"]

    if res["error"]:
        print(f"Error in {res['file']}: {res['error']}")
        return

    if res["rows"]:
        all_rows.extend(res["rows"])
        added_rows_count += len(res["rows"])
        added_rows_sec += sum(float(r.get("duration_sec", 0.0)) for r in res["rows"])

    added_src_count += 1
    files_since_ckpt += 1

    if total <= 20 or idx_done % 50 == 0:
        print(
            f"[{idx_done}/{total}] {res['file']}: "
            f"raw={res['raw_regions']} merged={res['merged_regions']} "
            f"packed={res['packed_regions']} final={res['final_segments']}"
        )


if pending_wavs:
    use_parallel = VAD_PARALLEL and VAD_NUM_WORKERS > 1 and len(pending_wavs) > 1
    if use_parallel:
        ExecutorCls = ProcessPoolExecutor if VAD_EXECUTOR == "process" else ThreadPoolExecutor
        max_in_flight = max(VAD_NUM_WORKERS, int(VAD_MAX_IN_FLIGHT))

        with ExecutorCls(max_workers=VAD_NUM_WORKERS) as ex:
            pending_iter = iter(pending_wavs)
            futures = {}

            # Prime the queue so workers stay busy without creating too many futures at once.
            for _ in range(min(max_in_flight, len(pending_wavs))):
                w = next(pending_iter, None)
                if w is None:
                    break
                futures[ex.submit(process_one_wav, w)] = w

            idx_done = 0
            pbar = tqdm(total=len(pending_wavs), desc="VAD files")
            try:
                while futures:
                    fut = next(as_completed(futures))
                    futures.pop(fut, None)
                    idx_done += 1
                    pbar.update(1)

                    res = fut.result()
                    handle_result(res, idx_done=idx_done, total=len(pending_wavs))

                    w = next(pending_iter, None)
                    if w is not None:
                        futures[ex.submit(process_one_wav, w)] = w

                    if CHECKPOINT_EVERY_FILES > 0 and files_since_ckpt >= CHECKPOINT_EVERY_FILES:
                        mdf = save_manifest(all_rows)
                        print(
                            f"Checkpoint: src_done_this_run={added_src_count} "
                            f"added_segments={added_rows_count} total_manifest_rows={len(mdf)}"
                        )
                        files_since_ckpt = 0
            finally:
                pbar.close()
    else:
        for idx_done, wav_path in enumerate(tqdm(pending_wavs, desc="VAD files"), start=1):
            res = process_one_wav(wav_path)
            handle_result(res, idx_done=idx_done, total=len(pending_wavs))

            if CHECKPOINT_EVERY_FILES > 0 and files_since_ckpt >= CHECKPOINT_EVERY_FILES:
                mdf = save_manifest(all_rows)
                print(
                    f"Checkpoint: src_done_this_run={added_src_count} "
                    f"added_segments={added_rows_count} total_manifest_rows={len(mdf)}"
                )
                files_since_ckpt = 0
else:
    print("No pending wavs. Resume state already complete.")

# Final save
final_df = save_manifest(all_rows)
print("Saved:", VAD_MANIFEST)
print("Total manifest segments:", len(final_df))

# Duration summary
raw_total_sec_all = 0.0
raw_total_sec_pending = 0.0
for w in wavs_all:
    try:
        raw_total_sec_all += float(sf.info(str(w)).duration)
    except Exception:
        pass
for w in pending_wavs:
    try:
        raw_total_sec_pending += float(sf.info(str(w)).duration)
    except Exception:
        pass

vad_total_sec_cum = float(final_df["duration_sec"].sum()) if not final_df.empty else 0.0

print("")
print("=== VAD Resume Summary ===")
print("Raw input hours (all)           :", round(raw_total_sec_all / 3600.0, 3))
print("Raw input hours (this run scope):", round(raw_total_sec_pending / 3600.0, 3))
print("Added src files this run        :", added_src_count)
print("Added segments this run         :", added_rows_count)
print("Added VAD hours this run        :", round(added_rows_sec / 3600.0, 3))
print("Cumulative VAD hours (manifest) :", round(vad_total_sec_cum / 3600.0, 3))

if pending_wavs and added_rows_count > 0:
    print("")
    print("=== VAD Duration Debug (this run only) ===")
    print("Raw speech hours (Ten-VAD)     :", round(raw_speech_sec_total / 3600.0, 3))
    print("Merged speech hours            :", round(merged_speech_sec_total / 3600.0, 3))
    print("After pad hours                :", round(padded_speech_sec_total / 3600.0, 3))
    print("After short-pack hours         :", round(packed_speech_sec_total / 3600.0, 3))
    print("Dropped by MIN_SEG_SEC hours   :", round(dropped_short_sec_total / 3600.0, 3))
    print("Kept after length-filter hours :", round(kept_after_lenfilter_sec_total / 3600.0, 3))
    print("Saved VAD hours (added)        :", round(added_rows_sec / 3600.0, 3))
    print("Dropped short regions count    :", dropped_short_regions_total)
    print("Raw/Merged region counts       :", raw_speech_regions_total, "/", merged_speech_regions_total)
    print("Padded/Packed region counts    :", padded_regions_total, "/", packed_regions_total)

    print("")
    print("Estimated kept hours by MIN_SEG_SEC (after short-pack, this run):")
    for m in [4, 5, 6, 8, 10]:
        est_sec = sum(d for d in packed_region_durations if d >= m)
        print(f"  >= {m}s : {round(est_sec / 3600.0, 3)} h")





